The previous topics covered what Docker is and how it works internally. Now we use it. The `docker run` command is where everything comes together — and it has enough flags and behaviours that it deserves a focused notebook.

In this notebook we'll cover:
- The anatomy of `docker run` and its most important flags
- Interactive mode vs detached mode
- Naming, port mapping, and passing environment variables
- The container lifecycle: create → start → stop → remove
- Executing commands in a running container with `docker exec`
- Streaming logs, copying files, and inspecting container state

## The `docker run` Command

`docker run` is actually shorthand for three separate operations:

```
docker run [OPTIONS] IMAGE [COMMAND] [ARG...]
    │
    ├── docker pull IMAGE       (if image not already local)
    ├── docker create IMAGE     (allocate container, writable layer)
    └── docker start CONTAINER  (execute the entry point process)
```

Most of the time you use `docker run` directly. The lower-level `docker create` + `docker start` split is useful when you need to configure something (like attaching a volume) before the process starts.

**Most important flags at a glance:**

| Flag | Short | Meaning |
|---|---|---|
| `--interactive --tty` | `-it` | Keep stdin open and allocate a pseudo-TTY (for interactive shells) |
| `--detach` | `-d` | Run in the background; print the container ID |
| `--name` | | Give the container a human-readable name |
| `--publish host:container` | `-p` | Map a host port to a container port |
| `--env KEY=VALUE` | `-e` | Set an environment variable inside the container |
| `--env-file FILE` | | Load environment variables from a file |
| `--rm` | | Automatically remove the container when it exits |
| `--volume host:container` | `-v` | Mount a host path or named volume into the container |
| `--network NAME` | | Connect to a specific Docker network |
| `--memory 512m` | `-m` | Set a memory limit |
| `--cpus 1.5` | | Limit to 1.5 CPU cores |
| `--workdir /path` | `-w` | Set the working directory inside the container |
| `--user uid:gid` | `-u` | Run as a specific user |
| `--entrypoint CMD` | | Override the image's default entrypoint |

## Interactive vs Detached Mode

### Interactive mode (`-it`)

Use `-it` when you want a shell or REPL inside the container — your terminal stays attached.

```bash
docker run -it ubuntu:22.04 bash
# → you get a bash prompt inside the container
# → Ctrl-D or 'exit' to leave (container stops when the shell exits)
```

`-i` keeps stdin open so you can type. `-t` allocates a pseudo-TTY so the shell prompt renders correctly. They are almost always used together.

### Detached mode (`-d`)

Use `-d` for long-running services — the container runs in the background and you get the container ID printed:

```bash
docker run -d --name webserver nginx:alpine
# → 3f9a2c1b7e4d...  (container ID, then your prompt returns)
```

### The `--rm` flag

Add `--rm` to have the container deleted automatically when it exits. Ideal for one-shot tasks where you don't need to inspect the stopped container afterwards:

```bash
docker run --rm python:3.12-slim python -c "print(2 ** 32)"
# runs, prints 4294967296, container is gone
```

> `--rm` and `-d` are mutually exclusive — a detached container runs until explicitly stopped, so auto-removal on exit doesn't apply the same way. Docker 25+ does support `--rm -d` together, removing the container when it eventually stops.

## Naming and Identifying Containers

Every container gets a unique **ID** (a 64-character hex string) at creation time. Docker also assigns a random human-readable **name** like `hopeful_lovelace` if you don't provide one.

```bash
docker run -d nginx:alpine
# 3f9a2c1b7e4d8b5c...  ← container ID

docker ps
# CONTAINER ID   IMAGE         ...  NAMES
# 3f9a2c1b7e4d   nginx:alpine  ...  hopeful_lovelace
```

Use `--name` to assign a meaningful name you can reference in scripts and other commands:

```bash
docker run -d --name webserver nginx:alpine
docker logs webserver       # use name, not ID
docker stop webserver
docker rm webserver
```

Names must be unique among all containers (including stopped ones). If you reuse a name without removing the old container first, Docker will error. A common pattern:

```bash
docker rm -f webserver 2>/dev/null; docker run -d --name webserver nginx:alpine
```

## Port Mapping

A container runs in its own network namespace — its ports are **not reachable from the host** unless you explicitly map them.

```bash
docker run -p HOST_PORT:CONTAINER_PORT image
```

```
Your browser  →  localhost:8080  →  Docker  →  container:80  →  Nginx
                 ──────────────     ──────     ─────────────
                   host port         NAT         container port
```

Common patterns:

| Flag | Effect |
|---|---|
| `-p 8080:80` | Host port 8080 → container port 80 |
| `-p 127.0.0.1:8080:80` | Bind only on localhost (not exposed to the network) |
| `-p 80` | Container port 80 → a random available host port |
| `-P` | Map all `EXPOSE`d ports to random host ports |

You can map multiple ports by repeating `-p`:

```bash
docker run -p 80:80 -p 443:443 nginx:alpine
```

To find which host port was assigned when using a random mapping:

```bash
docker port CONTAINER [CONTAINER_PORT]
```

## Environment Variables

Environment variables are the standard way to configure containers at runtime — database URLs, API keys, feature flags, log levels.

```bash
# Single variable
docker run -e LOG_LEVEL=debug myapp

# Multiple variables
docker run -e DB_HOST=postgres -e DB_PORT=5432 -e DB_NAME=appdb myapp

# Pass a variable from the host environment (no = sign → use host's value)
docker run -e AWS_ACCESS_KEY_ID myapp

# Load from a file (one KEY=VALUE per line)
docker run --env-file .env myapp
```

A `.env` file looks like:
```
DB_HOST=postgres
DB_PORT=5432
DB_NAME=appdb
# lines starting with # are comments
```

> **Security note:** avoid passing secrets via `-e` in production — they appear in `docker inspect` output and in shell history. Use Docker secrets, Vault, or a secrets manager instead. At minimum, use `--env-file` and keep the file out of version control.

## The Container Lifecycle

A container moves through a defined set of states:

```
          docker create
               │
               ▼
           [created]  ←─────────────────────────┐
               │                                 │
         docker start                            │
               │                                 │
               ▼                                 │
           [running] ──── docker pause ──► [paused]
               │                  ▲── docker unpause ──┘
     docker stop/kill
               │
               ▼
           [exited]  ◄── process exits on its own
               │
         docker start  (restart a stopped container)
               │
               ▼
           [running]
               │
          docker rm
               │
               ▼
           [gone]  (container and its writable layer deleted)
```

| Command | What it does |
|---|---|
| `docker create IMAGE` | Allocates the container and writable layer, does not start it |
| `docker start NAME` | Starts a created or stopped container |
| `docker stop NAME` | Sends SIGTERM, waits 10 s, then SIGKILL — graceful shutdown |
| `docker kill NAME` | Sends SIGKILL immediately — no grace period |
| `docker restart NAME` | stop + start |
| `docker pause NAME` | Freezes all processes in the container (SIGSTOP to the cgroup) |
| `docker unpause NAME` | Resumes frozen processes |
| `docker rm NAME` | Removes a stopped container (use `-f` to force-remove a running one) |
| `docker rm -f NAME` | Kill + remove in one step |

## Exec, Logs, and Copy

### `docker exec` — run a command in a running container

```bash
# Open an interactive shell in a running container
docker exec -it webserver sh

# Run a one-off command without attaching a shell
docker exec webserver nginx -t          # test nginx config
docker exec webserver cat /etc/hosts    # read a file
```

`exec` starts a new process inside the container's existing namespaces — it does not start a new container. The container must already be running.

### `docker logs` — view container output

```bash
docker logs webserver           # all output so far
docker logs -f webserver        # follow (stream) live output
docker logs --tail 50 webserver # last 50 lines
docker logs --since 5m webserver  # output from the last 5 minutes
```

Docker captures everything the container writes to stdout and stderr. Logs are stored by the daemon according to the configured log driver (default: `json-file`).

### `docker cp` — copy files between host and container

```bash
# Host → container
docker cp ./nginx.conf webserver:/etc/nginx/nginx.conf

# Container → host
docker cp webserver:/var/log/nginx/access.log ./access.log
```

`docker cp` works on both running and stopped containers. It is useful for one-off file transfers during debugging, but for persistent sharing, use bind mounts (covered in the Storage topic).

## Hands-On: Running and Managing Containers

In [ ]:
# Run a one-shot command — Python evaluates an expression, container is auto-removed
# --rm: delete container on exit   python:3.12-slim: the image   python -c: run inline code
!docker run --rm python:3.12-slim python -c "import sys; print('Python', sys.version)"

In [ ]:
# Run a detached Nginx web server with a port mapping and a name
# -d: detached (background)   --name: human-readable name   -p 8080:80: host:container
!docker run -d --name first-nginx -p 8080:80 nginx:alpine
!docker ps --filter name=first-nginx

In [ ]:
# Confirm the web server is reachable on the mapped host port
!curl -s -o /dev/null -w "HTTP status: %{http_code}\n" http://localhost:8080

In [ ]:
# Stream the container's logs — Nginx logs the curl request from the previous cell
!docker logs first-nginx

In [ ]:
# Run a command inside the already-running container without stopping it
# exec starts a new process in the container's namespaces
!docker exec first-nginx nginx -v                   # nginx version
!docker exec first-nginx cat /etc/nginx/nginx.conf  # nginx config

In [ ]:
# Pass environment variables — run a container that prints them back
!docker run --rm \
    -e APP_ENV=production \
    -e DB_HOST=postgres.internal \
    -e DB_PORT=5432 \
    alpine:3.19 \
    sh -c 'echo "APP_ENV=$APP_ENV  DB_HOST=$DB_HOST  DB_PORT=$DB_PORT"'

In [ ]:
# Copy a file from the running container to the host
!docker cp first-nginx:/etc/nginx/nginx.conf /tmp/nginx-from-container.conf
!echo "--- file copied from container ---"
!head -10 /tmp/nginx-from-container.conf

In [ ]:
# Inspect the container's current state via the daemon
import subprocess, json

raw = subprocess.check_output(["docker", "inspect", "first-nginx"])
data = json.loads(raw)[0]

state = data["State"]
net   = data["NetworkSettings"]["Ports"]

print(f"Status   : {state['Status']}")
print(f"PID      : {state['Pid']}")
print(f"Started  : {state['StartedAt']}")
print(f"Port map : {net}")

In [ ]:
# Stop the container gracefully (SIGTERM → 10 s grace → SIGKILL)
!docker stop first-nginx

# Confirm it is stopped — the port is no longer listening
!docker inspect --format '{{.State.Status}}' first-nginx

In [ ]:
# A stopped container still exists — you can restart it
!docker start first-nginx
!docker inspect --format '{{.State.Status}}' first-nginx

In [ ]:
# Show all containers — running and stopped — with their status
!docker ps -a --format 'table {{.Names}}\t{{.Image}}\t{{.Status}}\t{{.Ports}}'

In [ ]:
# Clean up: force-remove the container (stop + delete in one step)
!docker rm -f first-nginx
!echo "Done — container and its writable layer are deleted"

## Summary

- `docker run` is shorthand for `pull` (if needed) + `create` + `start`. Most flags you'll use daily are on `docker run`.
- **`-it`** keeps stdin open and allocates a TTY — use for interactive shells. **`-d`** runs the container in the background — use for services. **`--rm`** auto-deletes the container on exit — use for one-shot tasks.
- **`--name`** gives the container a human-readable handle. Names must be unique among all containers, including stopped ones.
- **`-p HOST:CONTAINER`** maps a host port to a container port — without it, the container's ports are not reachable from outside.
- **`-e KEY=VALUE`** sets environment variables; **`--env-file`** loads them from a file. Avoid passing secrets via `-e` in production.
- The container lifecycle: `created` → `running` → `exited` → removed. Stopped containers still exist until `docker rm`.
- **`docker stop`** sends SIGTERM first (graceful shutdown) then SIGKILL after 10 seconds. **`docker kill`** sends SIGKILL immediately.
- **`docker exec`** runs an additional process inside a running container — the container keeps running when you exit.
- **`docker logs`** captures everything written to stdout/stderr. Use `-f` to stream live output.
- **`docker cp`** copies files between host and container and works on stopped containers too.